In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client

In [2]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelGP/ModelGP_M05.json")
client.get_next_trials(max_trials=1)

{56: {'n_ci': 0.780417667206862, 'n_it': 0.3751109604213449}}

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 700

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.PseudorandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(client.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[17.033875882171934, 13.620661770789784, 13.769919269456322, 13.67613270571582, 15.799189288943014, 13.712037557926749, 14.512836472222254, 16.229721143830844, 14.190836369309691, 13.419333817895648, 14.85630231413944, 14.443327067002079, 14.208188364414847, 13.677541121084012, 13.893117399893313, 14.264491556603431, 13.487508060868961, 13.67004603962425, 14.484224125319876, 13.727487726600927, 15.590244846076136, 13.578406522373333, 15.727538181823416, 15.151663286495609, 13.502638671830937, 15.454146658485636, 15.405893254429392, 16.484412529885823, 14.104907879594819, 14.134701175798995, 14.640844344829137, 15.292896904892503, 14.061366130205673, 15.379006499947252, 15.82549830637857, 14.250256168336243, 16.18575649249901, 13.71903814103958, 14.452048756465327, 13.92023652799299, 14.370981171180894, 15.585919834558517, 14.034241970183391, 16.149853716135617, 15.31045484572422, 13.64470177912861, 13.840004851933621, 14.540438061991125, 17.349738780912915, 15.818169374714895, 13.81839

In [5]:
np.average(y_max_arr)

np.float64(14.73910676810604)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_M05/DataGenerated/pseudorandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)